# D3-02 · U-Net

### Steps
1. U-Net anatomy with tensor shapes
2. Write the encoder block, the decoder block and assemble the U-Net
3. Output activations and losses for binary vs multi-class masks (BCE + Dice)
4. Train on synthetic shapes in ~2 minutes; ablate the skip connections
5. Pet trimaps with a pretrained ResNet-18 encoder; read the errors
6. The family: U-Net++, Attention U-Net, DeepLabv3+, SegFormer, Mask2Former, diffusion U-Nets
7. Inference tricks: sliding windows, test-time flips, threshold tuning

### Topics covered and learning objectives
- Encoder (what) + decoder (where) + skips (fine detail) = U-Net
- Shape bookkeeping through 4 levels; why input sizes must be divisible by 2^depth
- Skip connections measured: same network, same budget, with and without
- Reusing an ImageNet encoder for dense prediction (discriminative learning rates)
- Error maps (false positives vs false negatives) instead of a single number
- Practical inference for big images and for binary decisions

### Time estimate
~90 min in class (guided + hands-on)

### Prerequisites
D3-01 (metrics, Dice, transposed convolution), Day 1 transfer learning.

## 0 · Setup

<details class="api-ref" data-api="np.set_printoptions,torch.manual_seed,torchkit.get_device"><summary><b>Functions from earlier notebooks, used again here</b> (click to open): <code>np.set_printoptions</code> · <code>torch.manual_seed</code> · <code>torchkit.get_device</code></summary>
<ul><li><code>np.set_printoptions(precision=2, suppress=True)</code> &nbsp; Changes how NumPy prints arrays for the rest of the session: precision=2 shows two decimals, suppress=True writes tiny numbers as 0. instead of in scientific notation (2e-05). Only the display changes, never the values.</li><li><code>torch.manual_seed(seed)</code> &nbsp; Fixes PyTorch&#x27;s random number generator, so random weights, shuffles and augmentations come out the same on every run. The notebooks call it in the setup cell and again right before creating a model, so that its initial weights are repeatable.</li><li><code>device = torchkit.get_device()</code> &nbsp; Course helper. Returns torch.device(&quot;cuda&quot;) when PyTorch can use an NVIDIA GPU, torch.device(&quot;mps&quot;) on an Apple-silicon Mac, otherwise torch.device(&quot;cpu&quot;), and prints which one it picked (for the CPU, with the number of threads PyTorch uses). Models and batches are then moved there with .to(device).</li></ul></details>

In [ ]:
import sys, pathlib
for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:   # find the cvcourse package
    if (p / "cvcourse").is_dir():
        sys.path.insert(0, str(p)); break

import time, inspect
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Patch
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader
from cvcourse import viz, seg, shapes, tests, torchkit, day3, labs

%matplotlib inline
plt.rcParams["figure.dpi"] = 90
np.set_printoptions(precision=3, suppress=True)
torch.manual_seed(0)
device = torchkit.get_device()
if device.type == "cpu":
    day3.tune_cpu_threads()

## 1 · Anatomy

U-Net (Ronneberger, Fischer & Brox, 2015) was designed for biomedical images with a few
dozen labelled examples. It is the D3-01 autoencoder plus one idea: **at every
resolution, copy the encoder features across and concatenate them to the decoder**.

* **Encoder** (contracting path): `[conv3x3-BN-ReLU] x 2`, then `maxpool 2x2`. Resolution halves,
  channels double: it learns *what* is in the image.
* **Bottleneck**: the coarsest, most semantic features (stride 16 for 4 levels).
* **Decoder** (expanding path): `2x2 transposed conv` (x2, no overlap, no checkerboard),
  **concatenate the skip** of the same resolution, `[conv3x3-BN-ReLU] x 2`.
* **Head**: a `1x1` conv to `num_classes` logits per pixel.

For a 128 px RGB input, 4 levels and 16 base channels:

In [ ]:
fig, ax = plt.subplots(figsize=(15, 6.2))
day3.draw_unet(ax, input_size=128, depth=4, base=16, in_ch=3, num_classes=4)
plt.tight_layout(); plt.show()

Follow the left column down: every step halves the side and doubles the channels; every orange skip
hands an encoder map to the decoder block of the **same** size.

### Lab · U-Net shapes
**Predict first:** this network has about 1.9 M parameters. Roughly what share of them sits in the
bottleneck alone, and which level needs the most activation memory during training? Read the two bar
charts, then work through the "Try this" list under the lab (5 minutes).

In [ ]:
labs.show("unet-shapes")

### Exercise 1 · the shape schedule

Return `[(channels, size)]` for each encoder level and the bottleneck (`depth + 1` tuples of
ints): level `i` has `base * 2**i` channels at `input_size // 2**i` (integer division, as
max-pooling floors). (The original paper used 64 base channels, unpadded convolutions and a
572 px input, the lab's "original 2015" preset; "same" padding makes the bookkeeping trivial.)

In [ ]:
def unet_shapes(input_size, depth, base):
    '''input_size (px), depth (number of poolings), base (channels of level 0) ->
    a list of depth + 1 tuples (channels, size): the encoder levels 0 .. depth-1, then the bottleneck.'''
    schedule = []
    for i in range(depth + 1):                     # level i; the last one (i = depth) is the bottleneck
        channels = base * 2 ** i                   # the channels double at every level
        size = input_size // 2 ** i                # the side halves (max-pooling floors)
        schedule.append((channels, size))
    return schedule

tests.check_unet_shapes(unet_shapes)

> **Divisibility.** Four poolings need a side divisible by $2^4 = 16$; otherwise the
> upsampled decoder map is one pixel smaller than its skip and `torch.cat` fails (the red
> concat you saw in the lab at 100 px: 24 vs 25). Pad the input (or crop the skip) when
> sizes do not divide.

## 2 · The building blocks

`day3.double_conv(in_ch, out_ch)` gives you the `[conv3x3-BN-ReLU] x 2` unit
(convolutions without bias, because BatchNorm adds its own shift).

In [ ]:
print(day3.double_conv(3, 16))

### Exercise 2 · `EncoderBlock` and `DecoderBlock`

* `EncoderBlock(in_ch, out_ch)`: `forward(x)` returns **two** tensors, `(pooled, skip)`:
  `skip = double_conv(x)` at full resolution and `pooled = maxpool2x2(skip)`.
* `DecoderBlock(in_ch, skip_ch, out_ch, use_skip=True)`: upsample `x` with
  `nn.ConvTranspose2d(in_ch, out_ch, 2, stride=2)`, concatenate the skip **on the channel
  axis**, then `double_conv(out_ch + skip_ch, out_ch)`. With `use_skip=False` the
  concatenation is skipped (the ablation in section 4) and the conv takes `out_ch` channels.

<details class="api-ref" data-api="nn.MaxPool2d,nn.ConvTranspose2d,torch.cat"><summary><b>Functions you will need</b> (click to open): <code>nn.MaxPool2d</code> · <code>nn.ConvTranspose2d</code> · <code>torch.cat</code></summary>
<ul><li><code>nn.MaxPool2d(kernel_size, stride=None)</code> &nbsp; Max pooling: keeps the largest value of every kernel_size x kernel_size window, separately for each channel; it has no parameters. The stride defaults to kernel_size, so nn.MaxPool2d(2) halves the height and width: (B, C, 28, 28) -&gt; (B, C, 14, 14).</li><li><code>nn.ConvTranspose2d(in_ch, out_ch, kernel_size, stride=1, padding=0, output_padding=0)</code> &nbsp; A learnable upsampling layer: the transposed convolution of F.conv_transpose2d (each input pixel paints a scaled copy of the kernel into the output; overlaps add up). Output side = (H - 1) * stride - 2 * padding + kernel_size + output_padding. kernel_size=2, stride=2 doubles H and W with no overlap at all: the upsampler of the U-Net decoder.</li><li><code>torch.cat(tensors, dim=0)</code> &nbsp; Joins a list of tensors along an existing dimension dim; all other dimensions must match. dim=0 appends samples (collect per-batch results into one tensor); dim=1 on (N, C, H, W) tensors puts channels side by side, as an Inception block does with its branches.</li></ul></details>

In [ ]:
class EncoderBlock(nn.Module):
    '''EncoderBlock(in_ch, out_ch): forward(x (N, in_ch, H, W)) -> (pooled (N, out_ch, H/2, W/2), skip (N, out_ch, H, W)).'''
    def __init__(self, in_ch, out_ch):
        super().__init__()
        # 1. two 3x3 conv-BN-ReLU
        self.conv = day3.double_conv(in_ch, out_ch)
        # 2. 2x2 max-pool
        self.pool = nn.MaxPool2d(2)

    def forward(self, x):                      # x: (N, in_ch, H, W)
        # 3. full resolution, kept for the decoder
        skip = self.conv(x)                    # (N, out_ch, H, W)
        # 4. half resolution, goes down to the next level
        pooled = self.pool(skip)               # (N, out_ch, H/2, W/2)
        # 5. both
        return pooled, skip


class DecoderBlock(nn.Module):
    '''DecoderBlock(in_ch, skip_ch, out_ch): forward(x (N, in_ch, h, w), skip (N, skip_ch, 2h, 2w)) -> (N, out_ch, 2h, 2w).'''
    def __init__(self, in_ch, skip_ch, out_ch, use_skip=True):
        super().__init__()
        self.use_skip = use_skip
        # 1. x2 upsampling, in_ch -> out_ch
        self.up = nn.ConvTranspose2d(in_ch, out_ch, kernel_size=2, stride=2)
        # 2. the channels after the concatenation
        if use_skip:
            conv_in = out_ch + skip_ch
        else:
            conv_in = out_ch
        # 3. two 3x3 conv-BN-ReLU
        self.conv = day3.double_conv(conv_in, out_ch)

    def forward(self, x, skip=None):           # x: (N, in_ch, h, w), skip: (N, skip_ch, 2h, 2w)
        # 4. upsample
        x = self.up(x)                         # (N, out_ch, 2h, 2w)
        # 5. glue the skip to it, on the channel axis
        if self.use_skip:
            x = torch.cat([x, skip], dim=1)    # (N, out_ch + skip_ch, 2h, 2w)
        # 6. mix them
        return self.conv(x)                    # (N, out_ch, 2h, 2w)

tests.check_unet_blocks(EncoderBlock, DecoderBlock)

### Exercise 3 · assemble the U-Net

Encoders with channels `base * 2**i` (`i < depth`), a bottleneck `double_conv` to
`base * 2**depth`, decoders in reverse order each fed the **matching skip**, and a 1x1
head. Keep the skips in a list on the way down and walk it backwards on the way up.
`forward(x)` takes `(N, in_ch, H, W)` and returns logits `(N, num_classes, H, W)`.

<details class="api-ref" data-api="nn.ModuleList,nn.Conv2d,torchkit.summarize"><summary><b>Functions from earlier notebooks, used again here</b> (click to open): <code>nn.ModuleList</code> · <code>nn.Conv2d</code> · <code>torchkit.summarize</code></summary>
<ul><li><code>nn.ModuleList([module, ...])</code> &nbsp; A list of layers that the model knows about: their parameters appear in model.parameters() (so the optimizer trains them) and move with model.to(device). Use it for a variable number of layers that your forward() calls in its own way, e.g. one lateral conv per pyramid level; unlike nn.Sequential it has no forward of its own.</li><li><code>nn.Conv2d(in_channels, out_channels, kernel_size, stride=1, padding=0, dilation=1, bias=True)</code> &nbsp; A convolution layer: out_channels learned kernels of size (in_channels, k, k), each slid over the input exactly like this notebook&#x27;s cross-correlation (no kernel flip). Input (B, C_in, H, W) -&gt; output (B, C_out, H_out, W_out), with H_out from the output-size formula. The weight has shape (C_out, C_in, k, k), the bias (C_out,).</li><li><code>print(torchkit.summarize(model, input_size))</code> &nbsp; Course helper. Pushes one dummy (all-zero) image of input_size, (C, H, W), through the model and returns a table as a string: one row per layer with its name, type, output shape, number of parameters and MACs (multiply-accumulates, counted for conv and linear layers), then the totals. The model&#x27;s train / eval mode is restored afterwards.</li></ul></details>

In [ ]:
class UNet(nn.Module):
    '''UNet(in_ch, num_classes, base, depth): forward(x (N, in_ch, H, W)) -> logits (N, num_classes, H, W);
    H and W must be divisible by 2 ** depth.'''
    def __init__(self, in_ch=3, num_classes=4, base=16, depth=4, use_skips=True):
        super().__init__()
        # 1. the channels of the encoder levels, e.g. [16, 32, 64, 128]
        chs = []
        for i in range(depth):
            chs.append(base * 2 ** i)
        # 2. the encoders, finest first
        self.encoders = nn.ModuleList()
        c_prev = in_ch
        for c in chs:
            self.encoders.append(EncoderBlock(c_prev, c))
            c_prev = c
        # 3. the bottleneck doubles the channels once more
        self.bottleneck = day3.double_conv(chs[-1], chs[-1] * 2)
        # 4. the decoders, coarsest first
        self.decoders = nn.ModuleList()
        for c in reversed(chs):                          # 128, 64, 32, 16
            self.decoders.append(DecoderBlock(c * 2, c, c, use_skips))
        # 5. one logit per class and pixel
        self.head = nn.Conv2d(base, num_classes, kernel_size=1)

    def forward(self, x):
        # 6. down: keep every skip
        skips = []
        for encoder in self.encoders:
            x, skip = encoder(x)                         # x: half the size; skip: full size
            skips.append(skip)
        # 7. the bottleneck
        x = self.bottleneck(x)
        # 8. up: the coarsest skip goes to the first decoder
        for decoder, skip in zip(self.decoders, reversed(skips)):
            x = decoder(x, skip)                         # twice the size
        # 9. the logits
        return self.head(x)                              # (N, num_classes, H, W)

tests.check_unet_forward(UNet)

In [ ]:
net = UNet(in_ch=3, num_classes=4, base=16, depth=4)
print(torchkit.summarize(net, (1, 3, 128, 128), depth=2))

Check the total against the lab (1,942,628 for this configuration) and find the bottleneck and the
deepest decoder block in the table: together they hold most of the weights.

### Output activation and loss

| task | head channels | activation at inference | loss | prediction |
|---|---|---|---|---|
| binary (pet vs not) | 1 | sigmoid | `BCEWithLogits` + Dice | `prob > threshold` (tune it, section 6) |
| multi-class (C classes, one per pixel) | C | softmax | `CrossEntropy` (+ class weights) + multi-class Dice | `argmax` |
| multi-label (overlapping masks) | C | sigmoid per channel | BCE per channel + Dice | `prob_c > threshold_c` |

Never put the sigmoid / softmax inside the model when training: the `...WithLogits`
and `cross_entropy` losses apply it in a numerically stable way (log-sum-exp).

### Exercise 4 · BCE + Dice

`bce_dice_loss(logits, target, bce_weight=0.5)` for binary masks:
`bce_weight * BCEWithLogits + (1 - bce_weight) * soft Dice of sigmoid(logits)`, with the
soft Dice of D3-01 (`eps = 1e-6`, over the whole batch).

<details class="api-ref" data-api="F.binary_cross_entropy_with_logits,torch.sigmoid,Tensor.float,Tensor.sum"><summary><b>Functions you will need</b> (click to open): <code>F.binary_cross_entropy_with_logits</code> · <code>torch.sigmoid</code> · <code>Tensor.float</code> · <code>Tensor.sum</code></summary>
<ul><li><code>F.binary_cross_entropy_with_logits(logits, targets, reduction=&#x27;mean&#x27;)</code> &nbsp; Binary cross-entropy for yes / no outputs, computed from raw logits z: per element -[t log sigmoid(z) + (1 - t) log(1 - sigmoid(z))], then the mean (reduction=&quot;none&quot; keeps one loss per element, &quot;sum&quot; adds them up). targets are floats 0. / 1. of the same shape. Objectness and focal loss use it.</li><li><code>torch.sigmoid(x)</code> &nbsp; Squashes every value (a logit) into (0, 1): sigmoid(x) = 1 / (1 + exp(-x)). Turns an objectness logit into a probability; D2-03 and D2-04 also use it to keep predicted box coordinates between 0 and 1.</li><li><code>x.float()</code> &nbsp; Returns the tensor converted to float32, the dtype layers and losses work in (the same as x.to(torch.float32)); x itself is not changed. A uint8 image becomes 0.0 to 255.0, so divide by 255 to get [0, 1].</li><li><code>x.sum(dim=None, keepdim=False)</code> &nbsp; Adds up all elements (giving a 0-dimensional tensor), or only along dim, which then disappears from the shape: for a (2, 3) tensor, sum(dim=0) adds the two rows together and gives one value per column, shape (3,). dim can be a tuple; keepdim=True keeps the summed dimension with size 1.</li></ul></details>

In [ ]:
def bce_dice_loss(logits, target, bce_weight=0.5):
    '''logits, target (0 / 1): same shape, one channel -> bce_weight * BCE + (1 - bce_weight) * soft Dice loss, a 0-d tensor.'''
    target = target.float()
    # 1. BCE on the logits (stable)
    bce = F.binary_cross_entropy_with_logits(logits, target)
    # 2. the probabilities
    p = torch.sigmoid(logits)
    # 3. the soft intersection
    inter = (p * target).sum()
    # 4. the soft Dice loss of D3-01
    dice_loss = 1 - (2 * inter + 1e-6) / (p.sum() + target.sum() + 1e-6)
    # 5. the weighted sum
    return bce_weight * bce + (1 - bce_weight) * dice_loss

tests.check_bce_dice(bce_dice_loss)

For our multi-class problems we use the same idea with softmax:
`day3.CEDiceLoss(dice_weight=0.5)` = `0.5 * cross_entropy + 0.5 * multi-class soft Dice`.

## 3 · Train on synthetic shapes (~2 minutes on a CPU)

We now have the network and the loss; what is missing is data and a training run, first on
synthetic shapes where the answer is known exactly.

64 px images with circles, squares and triangles (4 classes with background). Colour
already gives the class away here, so the real challenge is **where the boundary is** -
exactly what the skip connections are for. If your `UNet` did not pass the checker, use
`UNet = day3.UNet`.

<details class="api-ref" data-api="torchkit.ShapesDataset,torch.utils.data.DataLoader,Tensor.unique,Tensor.tolist,Tensor.to,torch.optim.AdamW,Module.parameters,torchkit.count_parameters,np.round"><summary><b>Functions from earlier notebooks, used again here</b> (click to open): <code>torchkit.ShapesDataset</code> · <code>torch.utils.data.DataLoader</code> · <code>Tensor.unique</code> · <code>Tensor.tolist</code> · <code>Tensor.to</code> · <code>torch.optim.AdamW</code> · <code>Module.parameters</code> · <code>torchkit.count_parameters</code> · <code>np.round</code></summary>
<ul><li><code>torchkit.ShapesDataset(n=512, size=128, seed=0, task=&quot;detection&quot;)</code> &nbsp; Course helper. n synthetic images of circles, squares and triangles, generated in memory from seed (no download, the same images every time). Items are (image, target) with a float image (3, size, size) in [0, 1]; task=&quot;detection&quot; gives {&quot;boxes&quot;: (N, 4) as x1, y1, x2, y2, &quot;labels&quot;: (N,)} with 0 circle, 1 square, 2 triangle; task=&quot;segmentation&quot; gives an int64 mask (size, size) with 0 background, 1 circle, 2 square, 3 triangle. The names are in ds.classes.</li><li><code>DataLoader(dataset, batch_size=1, shuffle=False, collate_fn=None)</code> &nbsp; Turns a dataset (anything with len(ds) and ds[i]) into batches you can loop over: each step takes batch_size items and stacks them, so images (C, H, W) become one tensor (B, C, H, W) and integer labels a tensor (B,). shuffle=True draws a new random order every epoch (for training; evaluation does not need it).</li><li><code>x.unique(return_counts=False)</code> &nbsp; The distinct values of a tensor, sorted, as a 1-D tensor; with return_counts=True also how often each one occurs. Handy to check which class ids a segmentation mask contains.</li><li><code>x.tolist()</code> &nbsp; Converts a tensor into (nested) Python lists of plain Python numbers, for printing a few values readably. A 0-dimensional tensor gives a single number, like .item().</li><li><code>x.to(device) / x.to(dtype) / model.to(device)</code> &nbsp; Returns the tensor converted to another device (x.to(&quot;cuda&quot;)), another dtype (x.to(torch.float32)) or both; if nothing changes, x itself. On a model, model.to(device) moves all its weights and returns the model. The course moves every model and every batch to the device picked by torchkit.get_device.</li><li><code>torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2)</code> &nbsp; Adam with decoupled weight decay, the default optimiser of modern training. Adam divides each parameter&#x27;s step by a running average of its own gradient size, so parameters move at a similar pace whatever the scale of their gradients; AdamW also shrinks every weight by the fraction lr * weight_decay at each step, which keeps the weights small and reduces overfitting. Used exactly like SGD.</li><li><code>model.parameters()</code> &nbsp; All learnable tensors of a model (the weights and biases of every layer inside it), one after the other. What you hand to an optimiser: torch.optim.SGD(model.parameters(), lr=0.1).</li><li><code>torchkit.count_parameters(model, trainable_only=False)</code> &nbsp; Course helper. The total number of parameters of a model, sum(p.numel() for p in model.parameters()); trainable_only=True counts only those with requires_grad (useful once layers are frozen for fine-tuning).</li><li><code>np.round(a, decimals=0)</code> &nbsp; Rounds every element to the given number of decimals. D2-01 cleans up the IoU thresholds 0.5, 0.55, ..., 0.95 made by np.arange with it: the floating-point steps of arange are not exact.</li></ul></details>

In [ ]:
kw = dict(min_r=5, max_r=14)                       # object radius for 64 px images
shp_tr = torchkit.ShapesDataset(1200, 64, seed=0, task="segmentation", **kw)
shp_va = torchkit.ShapesDataset(200, 64, seed=1, task="segmentation", **kw)
shp_tl = DataLoader(shp_tr, batch_size=16, shuffle=True)
shp_vl = DataLoader(shp_va, batch_size=50)
x0, y0 = shp_va[0]
print("image", tuple(x0.shape), "mask", tuple(y0.shape), "classes", y0.unique().tolist())

results = {}
for name, use_skips in [("with skips", True), ("without skips", False)]:
    torch.manual_seed(0)
    model = UNet(in_ch=3, num_classes=4, base=16, depth=4, use_skips=use_skips).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=3e-3, weight_decay=1e-4)
    print(f"--- {name}: {torchkit.count_parameters(model):,} parameters")
    hist = day3.train_segmentation(model, shp_tl, day3.CEDiceLoss(0.5), opt, epochs=10, max_minutes=1.25,
                                   val_loader=shp_vl, num_classes=4, log_every=0)
    iou, miou = day3.iou_from_confusion(day3.confusion_over_loader(model, shp_vl, 4))
    results[name] = dict(model=model, hist=hist, iou=iou, miou=miou)
    print(f"{name}: val mIoU {miou:.3f}   per class {np.round(iou, 3)}")

> **Expected** (CPU, 1.25 min each): with skips mIoU > 0.95 after one epoch (often 0.99+);
> without skips well below that, with the circles and triangles hit hardest. On a GPU
> both runs take a few seconds per epoch.

<details class="api-ref" data-api="torch.no_grad,Module.eval,Tensor.argmax,Tensor.cpu,Tensor.permute,Tensor.numpy,np.arange,np.nan_to_num,np.convolve,np.ones"><summary><b>Functions from earlier notebooks, used again here</b> (click to open): <code>torch.no_grad</code> · <code>Module.eval</code> · <code>Tensor.argmax</code> · <code>Tensor.cpu</code> · <code>Tensor.permute</code> · <code>Tensor.numpy</code> · <code>np.arange</code> · <code>np.nan_to_num</code> · <code>np.convolve</code> · <code>np.ones</code></summary>
<ul><li><code>with torch.no_grad(): ...</code> &nbsp; A with block in which autograd records nothing: results have requires_grad=False, and computing them takes less memory and time. Use it when you only evaluate a model (testing, plotting its predictions), never around the training step itself.</li><li><code>model.eval()</code> &nbsp; Puts the model and all its layers into evaluation mode: dropout is switched off and BatchNorm uses the running averages collected during training, so a prediction no longer depends on the rest of the batch. Returns the model, so get_model(...).eval() works in one line.</li><li><code>x.argmax(dim)</code> &nbsp; The position of the largest value along dim. For logits (B, num_classes), x.argmax(dim=1) is the predicted class of every sample, shape (B,). Without dim it gives the position in the flattened tensor.</li><li><code>x.cpu()</code> &nbsp; Returns a copy of the tensor in main memory (the CPU), or the tensor itself if it is already there. Needed before .numpy() or plotting when the result was computed on a GPU: x.detach().cpu().numpy().</li><li><code>x.permute(*dims)</code> &nbsp; Reorders the dimensions: the argument lists the old positions in their new order. No data is copied. Matplotlib and image files use (H, W, C); PyTorch layers use (C, H, W); permute converts.</li><li><code>x.numpy()</code> &nbsp; Returns the tensor&#x27;s data as a NumPy array without copying: both share the same memory, so changing one changes the other. Used to hand results to Matplotlib or to NumPy code.</li><li><code>np.arange(stop) / np.arange(start, stop, step, dtype=None)</code> &nbsp; Like Python&#x27;s range, as a NumPy array: from start (default 0) up to but not including stop. With reshape, the quickest way to build a small test array with distinct values.</li><li><code>np.nan_to_num(a, nan=0.0)</code> &nbsp; Returns a copy of a with every NaN replaced by 0 (or by the value given as nan=), and infinities by very large finite numbers. Used before code that cannot handle NaN, such as writing the values into a figure.</li><li><code>np.convolve(a, v, mode=&#x27;full&#x27;)</code> &nbsp; Slides v along a and sums the products at every position (a 1-D convolution). With v = np.ones(k) / k it is a moving average over k values; mode=&quot;valid&quot; keeps only the positions where v fits entirely, so len(a) - k + 1 values come out. D2-02 smooths noisy loss curves this way.</li><li><code>np.ones(shape)</code> &nbsp; An array of the given shape (a number or a tuple) filled with 1.0, e.g. a constant learning-rate schedule: learning rate / peak = 1 at every step.</li></ul></details>

In [ ]:
fig = plt.figure(figsize=(16, 7.5))
gs = fig.add_gridspec(4, 6, width_ratios=[1, 1, 1, 1, 0.15, 2.6])
xv, yv = next(iter(DataLoader(shp_va, batch_size=4)))
preds = {}                                          # "with skips" / "without skips" -> predicted classes (4, 64, 64)
with torch.no_grad():
    for name, r in results.items():
        model = r["model"].eval()
        logits = model(xv.to(device))               # (4, 4 classes, 64, 64)
        preds[name] = logits.argmax(1).cpu()        # the most likely class of every pixel
for i in range(4):
    row = [xv[i].permute(1, 2, 0).numpy(), day3.colorize(yv[i]), day3.colorize(preds["with skips"][i]),
           day3.colorize(preds["without skips"][i])]
    for j, im in enumerate(row):
        ax = fig.add_subplot(gs[i, j]); ax.imshow(im, interpolation="nearest"); ax.axis("off")
        if i == 0: ax.set_title(["image", "ground truth", "U-Net with skips", "U-Net WITHOUT skips"][j], fontsize=10)
ax = fig.add_subplot(gs[:2, 5])
names4 = ["background"] + shapes.CLASS_NAMES
xk = np.arange(4)
for k, (name, r) in enumerate(results.items()):
    ax.bar(xk + (k - 0.5) * 0.38, np.nan_to_num(r["iou"]), 0.38, label=f"{name} (mIoU {r['miou']:.2f})",
           color=["#4C78A8", "#E45756"][k])
ax.set_xticks(xk); ax.set_xticklabels(names4); ax.set_ylim(0, 1.15); ax.legend(fontsize=8, loc="upper right")
ax.set_title("per-class IoU on 200 validation images")
ax = fig.add_subplot(gs[2:, 5])
for k, (name, r) in enumerate(results.items()):
    l = np.convolve(r["hist"]["step_loss"], np.ones(10) / 10, mode="valid")
    ax.plot(l, label=name, color=["#4C78A8", "#E45756"][k])
ax.set_xlabel("step"); ax.set_ylabel("training loss (CE + Dice, smoothed)"); ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

Same encoder, same decoder, same budget. Without the skips the decoder must re-invent
every boundary from a **4x4** bottleneck (64 px / 16): it knows *that* there is a
circle in the top-left, not where its outline runs - blobby, misplaced masks, small
objects vanish. With the skips the 64x64 encoder features (edges, colours) are handed
straight to the last decoder block.

## 4 · Real data: Pet trimaps with a pretrained ResNet-18 encoder

Small data + a pretrained encoder is the default recipe (Day 1). We take torchvision's
ImageNet ResNet-18 and use its stages as the U-Net encoder: the stem gives a stride-2
skip, `layer1..layer3` give stride 4/8/16 skips and `layer4` (stride 32) is the
bottleneck. The decoder reuses the `DecoderBlock` from `day3`. Read it:

<details class="api-ref" data-api="Tensor.numel,torch.zeros"><summary><b>Functions from earlier notebooks, used again here</b> (click to open): <code>Tensor.numel</code> · <code>torch.zeros</code></summary>
<ul><li><code>x.numel()</code> &nbsp; The total number of elements of a tensor: the product of its shape. Summed over model.parameters() it gives the number of parameters of a model.</li><li><code>torch.zeros(*size, dtype=None, device=None)</code> &nbsp; A new tensor of the given shape filled with 0 (float32 unless dtype says otherwise). Used to allocate outputs and to build test inputs of a given shape.</li></ul></details>

In [ ]:
print(inspect.getsource(day3.ResNetUNet.forward))
m_tmp = day3.ResNetUNet(num_classes=3, pretrained=False)
n_encoder = 0                                       # parameters of the ResNet-18 encoder
for p in m_tmp.encoder_parameters():
    n_encoder += p.numel()
n_decoder = 0                                       # parameters of the decoder
for p in m_tmp.decoder_parameters():
    n_decoder += p.numel()
print(f"encoder {n_encoder:,} params (ImageNet), decoder {n_decoder:,} params (random init)")
with torch.no_grad():
    print("output:", tuple(m_tmp(torch.zeros(1, 3, 128, 128)).shape))

In [ ]:
try:
    import segmentation_models_pytorch as smp
    lib = smp.Unet("resnet18", encoder_weights=None, classes=3)
    print("segmentation_models_pytorch is installed: smp.Unet('resnet18', encoder_weights='imagenet', classes=3) "
          f"would replace our hand-written model ({torchkit.count_parameters(lib):,} params). "
          "It also offers U-Net++, DeepLabv3+, FPN, PSPNet, SegFormer heads with 100+ encoders.")
except ImportError:
    print("segmentation_models_pytorch is not installed (optional). With it, one line replaces ResNetUNet:\n"
          "    smp.Unet('resnet18', encoder_weights='imagenet', classes=3)\n"
          "and the same API gives U-Net++, DeepLabv3+, FPN, SegFormer heads on 100+ encoders.")

**Training recipe**: 128 px, random resized crops + horizontal flips applied to image and
mask together (`torchvision.transforms.v2` understands `Mask` targets), CE + Dice loss,
AdamW with a 5x smaller learning rate on the pretrained encoder.

The cell trains for at most 3 minutes (1 200 images) when no checkpoint exists. The
course provides `data/checkpoints/d3_pet_resnet_unet.pt`, trained the same way on
2 000 training images for 3 epochs (OneCycle schedule, ~40 min on a CPU). Set
`FORCE_TRAIN = True` to train yourself.

> **Expected numbers.** Checkpoint: mIoU ~0.73 on 300 Pet test-split images at 128 px
> (background ~0.89, pet ~0.81, border ~0.50). A 3-minute CPU run on 1 200 images:
> mIoU ~0.6. On a GPU, 10 epochs on all 3 680 images at 224 px take ~5 minutes and reach ~0.80.

<details class="api-ref" data-api="torchkit.get_pet_datasets,torchkit.fit,torchkit.load_checkpoint,torchkit.denormalize,torch.Generator"><summary><b>Functions from earlier notebooks, used again here</b> (click to open): <code>torchkit.get_pet_datasets</code> · <code>torchkit.fit</code> · <code>torchkit.load_checkpoint</code> · <code>torchkit.denormalize</code> · <code>torch.Generator</code></summary>
<ul><li><code>train, val, classes = torchkit.get_pet_datasets(size=224, target=&quot;category&quot;, subset=None)</code> &nbsp; Course helper. Returns (train, val, classes) for Oxford-IIIT Pet (run tools/download_data.py first); images are float (3, size, size), resized and normalised with the ImageNet mean and std. target chooses the label: &quot;category&quot; gives (image, breed index 0..36) and the 37 breed names; &quot;segmentation&quot; gives (image, int64 mask (H, W)) with 0 background, 1 pet, 2 border; &quot;detection&quot; gives (image, {&quot;boxes&quot;: (N, 4) as x1, y1, x2, y2 in pixels, &quot;labels&quot;: (N,)}) for the pet&#x27;s head. subset=N keeps a class-balanced N training (and N // 4 validation) images for quick experiments.</li><li><code>history = torchkit.fit(model, train_loader, val_loader, loss_fn, optimizer, epochs, scheduler=None, device=None, max_minutes=None, checkpoint=None, restore_best=False, log_every=50)</code> &nbsp; Course helper: the training loop. It moves the model and every batch to device (default: where the model&#x27;s weights already are) and every epoch runs, batch by batch, forward, loss, optimizer.zero_grad(), backward(), optimizer.step() (plus scheduler.step(): after each batch for OneCycleLR, after each epoch for most other schedules), then evaluates on val_loader and prints one line. It returns a history dict of per-epoch lists, &quot;train_loss&quot;, &quot;val_loss&quot;, &quot;val_metric&quot; (accuracy for classifiers, NaN otherwise), &quot;lr&quot; and &quot;time&quot;, plus &quot;best_epoch&quot; and &quot;stopped_early&quot;.</li><li><code>extras = torchkit.load_checkpoint(model, path)</code> &nbsp; Course helper. Loads the weights saved in path (by torchkit.fit(checkpoint=path) or torchkit.save_checkpoint) into model, in place, and returns everything else stored in the file as a dict; for a fit checkpoint &quot;epoch&quot;, &quot;val_loss&quot;, &quot;val_metric&quot; and the whole &quot;history&quot;. The course keeps checkpoints in torchkit.CHECKPOINTS, which is data/checkpoints/.</li><li><code>torchkit.denormalize(x, mean=IMAGENET_MEAN, std=IMAGENET_STD)</code> &nbsp; Course helper. Undoes the normalisation (x * std + mean, per channel) and returns a NumPy array ready for plt.imshow, clipped to [0, 1]: (C, H, W) becomes (H, W, C), a batch (B, C, H, W) becomes (B, H, W, C), a 1-channel image becomes (H, W). The default statistics are ImageNet&#x27;s (torchkit.IMAGENET_MEAN, IMAGENET_STD); for FashionMNIST pass torchkit.FASHION_MNIST_MEAN, FASHION_MNIST_STD.</li><li><code>g = torch.Generator().manual_seed(seed)</code> &nbsp; A random number generator of your own. Passed as generator=g to torch.rand, torch.randn, torch.randperm or a DataLoader, it makes exactly those numbers repeatable, without touching (or being disturbed by) the global seed of torch.manual_seed.</li></ul></details>

In [ ]:
FORCE_TRAIN = False
CKPT = torchkit.CHECKPOINTS / "d3_pet_resnet_unet.pt"
train_tf, eval_tf = day3.pet_seg_transforms(128)
pet_tr, pet_va, pet_classes = torchkit.get_pet_datasets(size=128, target="segmentation", subset=(1200, 300),
                                                        train_tf=train_tf, eval_tf=eval_tf)
pet_tl = DataLoader(pet_tr, batch_size=16, shuffle=True)
pet_vl = DataLoader(pet_va, batch_size=32)

torch.manual_seed(0)
model_pet = day3.ResNetUNet(num_classes=3).to(device)
if CKPT.exists() and not FORCE_TRAIN:
    extra = torchkit.load_checkpoint(model_pet, CKPT)
    print(f"loaded {CKPT.name} (trained {extra.get('epoch', '?')} epochs; stored val metric {extra.get('val_metric', float('nan')):.3f})")
else:
    opt = day3.pet_unet_optimizer(model_pet)
    hist_pet = torchkit.fit(model_pet, pet_tl, pet_vl, day3.CEDiceLoss(0.5), opt, epochs=2,
                            metric_fn=day3.miou_metric(3), max_minutes=3, log_every=20, checkpoint=CKPT)
    torchkit.load_checkpoint(model_pet, CKPT)
model_pet.eval()
cm_pet = day3.confusion_over_loader(model_pet, pet_vl, 3)
iou_pet, miou_pet = day3.iou_from_confusion(cm_pet)
print(f"Pet validation ({len(pet_va)} test-split images, 128 px): mIoU {miou_pet:.3f}  " +
      "  ".join(f"{c} {v:.3f}" for c, v in zip(pet_classes, iou_pet)))

In [ ]:
xb, yb = next(iter(DataLoader(pet_va, batch_size=6, shuffle=True, generator=torch.Generator().manual_seed(3))))
with torch.no_grad():
    logits = model_pet(xb.to(device)).cpu()
pred = logits.argmax(1)
fig, axes = plt.subplots(4, 6, figsize=(15, 10.5))
for j in range(6):
    im = torchkit.denormalize(xb[j])
    axes[0, j].imshow(im)
    axes[1, j].imshow(day3.colorize(yb[j]))
    axes[2, j].imshow(day3.colorize(pred[j]))
    fg_p, fg_t = (pred[j] > 0).numpy().astype(int), (yb[j] > 0).numpy().astype(int)
    axes[3, j].imshow(day3.error_map(fg_p, fg_t, 1))
    iou_j = (fg_p & fg_t).sum() / max((fg_p | fg_t).sum(), 1)
    axes[3, j].set_title(f"foreground IoU {iou_j:.2f}", fontsize=9)
for r, lab in enumerate(["image", "ground truth", "prediction", "errors"]):
    axes[r, 0].set_ylabel(lab, fontsize=11)
for ax in axes.flat: ax.set_xticks([]); ax.set_yticks([])
fig.legend(handles=[Patch(color=day3.SEG_COLORS[i], label=c) for i, c in enumerate(pet_classes)] +
           [Patch(color=(0.75, 0.75, 0.75), label="correct foreground"), Patch(color=(0.9, 0.15, 0.15), label="false positive"),
            Patch(color=(0.15, 0.35, 0.95), label="false negative")], loc="lower center", ncol=6, fontsize=9)
plt.tight_layout(rect=(0, 0.04, 1, 1)); plt.show()

Read the error row, not the score: **blue** (missed pet) concentrates on thin parts -
ears, tails, legs - and on dark fur against dark backgrounds; **red** (hallucinated pet)
appears on blankets, cushions and people's arms, i.e. texture that co-occurs with pets
in the training set. The border class is the hardest by construction (it is a 2-3 pixel
band at 128 px, and annotators disagree on it).

## 5 · The U-Net family

Every modern segmentation network is "an encoder for *what*, a decoder for *where*,
and some way to bring fine detail back". They differ in the encoder (CNN or
transformer), how they gather context (ASPP, attention) and how they fuse scales.

In [ ]:
day3.draw_variants(plt.figure(figsize=(16, 9.5)))
plt.show()

| model | year | idea | when to use |
|---|---|---|---|
| U-Net | 2015 | symmetric encoder-decoder + skips | small data, medical, the default baseline |
| U-Net++ | 2018 | nested dense skips, deep supervision | when boundaries matter and compute is cheap |
| Attention U-Net | 2018 | gates on the skips | cluttered backgrounds (medical) |
| DeepLabv3+ | 2018 | dilated encoder + ASPP + one skip | natural scenes, strong CNN baseline |
| SegFormer | 2021 | hierarchical ViT encoder + MLP decoder | best accuracy/compute on ADE20K / Cityscapes class |
| Mask2Former | 2022 | queries -> (class, mask) pairs | one model for semantic, instance and panoptic |
| SAM / SAM 2 | 2023-24 | promptable, class-agnostic masks | D3-04: labelling, interactive tools |

**U-Net in diffusion models.** The denoiser of DDPM / Stable Diffusion is a U-Net: it
takes a noisy image $x_t$ and predicts the noise, *same size in, same size out* - a
dense prediction task like segmentation. Two additions: the timestep $t$ is embedded
(sinusoid -> MLP) and **added** to the features inside every residual block, and
self-attention / cross-attention (to the text prompt) layers sit at the low-resolution
levels. The skip connections are there for the same reason as here: fine detail.

## 6 · Inference tricks

### Sliding windows for big images

Satellite tiles, whole-slide pathology images and 4K frames do not fit in memory at
once. Predict on overlapping windows and **average the logits** where windows overlap
(the overlap hides the weaker predictions near window borders, where the network saw
less context).

### Lab · sliding-window inference
**Predict first:** a 200 x 200 image, 64 px windows, stride 48. How many forward passes does
`sliding_window_predict` run, and how many with stride 16? Check in the lab, then work through the
"Try this" list (4-5 minutes).

In [ ]:
labs.show("sliding-window")

### Exercise 5 · `sliding_window_predict`

`image (C, H, W)`; `model_fn` maps `(1, C, window, window)` to `(1, K, window, window)`.
Window positions `0, stride, 2*stride, ...`; if the last window does not end exactly at
the border, **add one more window ending at the border** (the dashed, shifted tiles in the
lab). Accumulate the outputs and a count map (the lab's "tiles per pixel"), return
`out / count` of shape `(K, H, W)`, with `K` taken from the model's output.

In [ ]:
def sliding_window_predict(model_fn, image, window, stride):
    '''model_fn: (1, C, window, window) -> (1, K, window, window); image (C, H, W) ->
    (K, H, W): every pixel is the average of the predictions of all windows that cover it.'''
    C, H, W = image.shape
    # 1. the top rows and left columns of the windows: 0, stride, 2 * stride, ...
    ys = list(range(0, max(H - window, 0) + 1, stride))
    xs = list(range(0, max(W - window, 0) + 1, stride))
    # 2. one more window that ends exactly at the bottom / right border, if needed
    if ys[-1] + window < H:
        ys.append(H - window)
    if xs[-1] + window < W:
        xs.append(W - window)
    # 3. the sums of the predictions (created at the first window, once K is known), and how many windows cover each pixel
    out = None
    count = torch.zeros(1, H, W)
    for y in ys:
        for x in xs:
            crop = image[:, y:y + window, x:x + window]       # (C, window, window)
            pred = model_fn(crop[None])[0]                    # (K, window, window)
            if out is None:
                out = torch.zeros(pred.shape[0], H, W, dtype=pred.dtype)   # (K, H, W)
            out[:, y:y + window, x:x + window] += pred
            count[:, y:y + window, x:x + window] += 1
    # 4. the average over the windows
    return out / count                                        # (K, H, W)

tests.check_sliding_window(sliding_window_predict)

<details class="api-ref" data-api="np.random.default_rng,torch.tensor"><summary><b>Functions from earlier notebooks, used again here</b> (click to open): <code>np.random.default_rng</code> · <code>torch.tensor</code></summary>
<ul><li><code>rng = np.random.default_rng(seed)</code> &nbsp; Creates a NumPy random number generator; the same seed gives the same numbers on every run. Draw from it with rng.normal(size=...) (normal distribution), rng.random(size) (uniform in [0, 1)), rng.integers(low, high, size) or rng.permutation(n).</li><li><code>torch.tensor(data, dtype=None, device=None)</code> &nbsp; Builds a new tensor from a Python number, a (nested) list or a NumPy array, always copying the data. The dtype is inferred: integers give int64, floats give float32; pass dtype= to choose.</li></ul></details>

In [ ]:
rng = np.random.default_rng(5)
big_img, _, _, big_mask = shapes.make_sample(rng, 200, max_objects=14, min_r=5, max_r=14)   # 200 px, not a multiple of 16
big = torch.tensor(big_img).permute(2, 0, 1)
shp_model = results["with skips"]["model"].eval()

def shapes_model_fn(window_batch):
    # one window (1, 3, 64, 64) -> logits (1, 4, 64, 64), back on the CPU
    return shp_model(window_batch.to(device)).cpu()

with torch.no_grad():
    logits_sw = sliding_window_predict(shapes_model_fn, big, window=64, stride=48)
pred_sw = logits_sw.argmax(0).numpy()
fig, axes = plt.subplots(1, 3, figsize=(14, 4.8))
axes[0].imshow(big_img)
for y in [0, 48, 96, 136]:
    for x in [0, 48, 96, 136]:
        axes[0].add_patch(Rectangle((x, y), 64, 64, fill=False, ec=viz.color((x + y) // 48), lw=1, alpha=0.8))
axes[0].set_title("200x200 image, 16 windows of 64 px (stride 48)")
axes[1].imshow(day3.colorize(big_mask)); axes[1].set_title("ground truth")
axes[2].imshow(day3.colorize(pred_sw)); axes[2].set_title(f"sliding-window prediction, mIoU {seg.mean_iou(pred_sw, big_mask, 4):.3f}")
for ax in axes: ax.axis("off")
plt.tight_layout(); plt.show()

These are the 16 windows of the lab's default state. The stitched prediction shows no tile seams
although no single window saw the whole image; a Gaussian weighting of each window (the lab's
"Gaussian" blend, used by nnU-Net and MONAI) would down-weight the window borders further.

### Test-time augmentation (TTA)

Predict on the image and its horizontal flip, flip the second prediction back, average.
Cheap, typically +0.5-1 mIoU point. (Older pipelines added a **CRF** on top to snap
boundaries to image edges; modern decoders made it mostly unnecessary.)

<details class="api-ref" data-api="Tensor.flip"><summary><b>Functions used here</b> (click to open): <code>Tensor.flip</code></summary>
<p><b><code>x.flip(dims)</code></b><br>Reverses the order of the elements along the given dimension(s) and returns a copy. On an (N, C, H, W) batch, x.flip(-1) mirrors every image left-right (the W axis). Flip test-time augmentation predicts on the flipped image and flips the prediction back.</p><pre>&gt;&gt;&gt; x = torch.tensor([[1, 2, 3],
...                   [4, 5, 6]])
&gt;&gt;&gt; x.flip(-1)                                   # last axis: left-right mirror
tensor([[3, 2, 1],
        [6, 5, 4]])
&gt;&gt;&gt; x.flip(0)                                    # first axis: upside down
tensor([[4, 5, 6],
        [1, 2, 3]])</pre><p><i>Watch out:</i> Tensors do not support negative slicing steps: x[..., ::-1] raises an error, x.flip(-1) is the way.</p><p>See also: <code>torchvision.transforms.v2.RandomHorizontalFlip</code></p><p><small>Any of these again later: <code>apiref.show("name")</code>.</small></p></details>

<details class="api-ref" data-api="Tensor.softmax,np.zeros,np.concatenate"><summary><b>Functions from earlier notebooks, used again here</b> (click to open): <code>Tensor.softmax</code> · <code>np.zeros</code> · <code>np.concatenate</code></summary>
<ul><li><code>logits.softmax(dim)</code> &nbsp; Turns scores (logits) into probabilities along dimension dim: exp(x) / sum(exp(x)). Every value ends up between 0 and 1 and they sum to 1 along dim; the order of the scores is kept, so the argmax does not change.</li><li><code>np.zeros(shape, dtype=float)</code> &nbsp; A new array of the given shape filled with 0.0 (float64). The standard way to allocate an output array before filling it in a loop.</li><li><code>np.concatenate(arrays, axis=0)</code> &nbsp; Joins arrays end to end along an EXISTING axis; all other dimensions must match. axis=0 appends rows; axis=1 puts columns side by side, e.g. top-left corners (N, 2) and bottom-right corners (N, 2) into xyxy boxes (N, 4). The AP code uses it to pad a curve with one value at each end.</li></ul></details>

In [ ]:
@torch.no_grad()                                    # no gradients inside this function
def predict_tta(model, x):
    prob = model(x).softmax(1)                      # (N, K, H, W) on the images
    prob_mirror = model(x.flip(-1)).softmax(1)      # on the mirrored images (flip the width axis)
    prob_back = prob_mirror.flip(-1)                # mirror those predictions back
    return 0.5 * (prob + prob_back)                 # the average of the two

cm_tta = np.zeros((3, 3), np.int64)
probs_fg = []                                       # P(foreground) = 1 - P(background), per batch
gts_fg = []                                         # the true foreground, per batch
for xb_, yb_ in pet_vl:
    p = predict_tta(model_pet, xb_.to(device)).cpu()
    cm_tta += seg.confusion_matrix(p.argmax(1).numpy(), yb_.numpy(), 3)
    probs_fg.append(1 - p[:, 0].numpy())
    gts_fg.append((yb_ > 0).numpy())
probs_fg = np.concatenate(probs_fg)                 # (300, 128, 128)
gts_fg = np.concatenate(gts_fg)
print(f"mIoU without TTA {miou_pet:.4f}  ->  with flip TTA {day3.iou_from_confusion(cm_tta)[1]:.4f}")

### Exercise 6 · tune the threshold on the validation set

Many applications only need **pet vs not pet**. The foreground probability is
`1 - p(background)`; `argmax` implicitly thresholds it near 0.5, but the IoU-optimal
threshold is rarely 0.5. Implement `threshold_sweep(prob, target, thresholds)` returning
`(best_threshold, ious)` where `ious[k]` is the IoU of `prob >= thresholds[k]` with the
target (over all pixels of all images). In a real project you would tune on validation
and report on a separate test set.

<details class="api-ref" data-api="np.logical_or"><summary><b>Functions you will need</b> (click to open): <code>np.logical_or</code></summary>
<p><b><code>np.logical_or(a, b)</code></b><br>Element-wise OR of two boolean (or 0/1) arrays of the same shape: True where at least one is true. On a predicted and a true mask it is their union, the denominator of IoU.</p><pre>&gt;&gt;&gt; pred = np.array([1, 1, 0, 0], bool)
&gt;&gt;&gt; gt   = np.array([0, 1, 1, 0], bool)
&gt;&gt;&gt; np.logical_or(pred, gt)
array([ True,  True,  True, False])
&gt;&gt;&gt; int(np.logical_and(pred, gt).sum()) / int(np.logical_or(pred, gt).sum())   # IoU = 1 / 3
0.3333333333333333</pre><p>See also: <code>np.logical_and</code></p><p><small>Any of these again later: <code>apiref.show("name")</code>.</small></p></details>

<details class="api-ref" data-api="np.logical_and,np.asarray,np.array,np.linspace,np.argmin,np.abs,Tensor.max"><summary><b>Functions from earlier notebooks, used again here</b> (click to open): <code>np.logical_and</code> · <code>np.asarray</code> · <code>np.array</code> · <code>np.linspace</code> · <code>np.argmin</code> · <code>np.abs</code> · <code>Tensor.max</code></summary>
<ul><li><code>np.logical_and(a, b)</code> &nbsp; Element-wise AND of two boolean (or 0/1) arrays of the same shape: True where both are true. On a predicted and a true mask it is their intersection, and .sum() counts its pixels; the same as a &amp; b for boolean arrays.</li><li><code>np.asarray(obj)</code> &nbsp; Converts its argument (a list, a PIL image, a CPU tensor, ...) to a NumPy array; an array is returned as it is, without a copy. A grey PIL image becomes a (H, W) uint8 array, a colour one (H, W, 3).</li><li><code>np.array(data, dtype=None)</code> &nbsp; Creates a NumPy array from a (nested) list, copying the data. The dtype is inferred: integers give int64, any float gives float64; dtype=float gives floats from integer literals.</li><li><code>np.linspace(start, stop, num=50)</code> &nbsp; num evenly spaced numbers from start to stop, both ends included. Used for plot axes and grids.</li><li><code>np.argmin(a, axis=None)</code> &nbsp; The position of the smallest value (the first one if several are equal). Without axis the array is treated as one flat list; with axis=k you get one position per row / column. Typical use: the epoch with the lowest validation loss.</li><li><code>np.abs(a)</code> &nbsp; The absolute value of every element. np.abs(a - b).max() is the largest difference between two arrays: the course&#x27;s check that two implementations compute the same numbers (up to tiny rounding differences).</li><li><code>x.max() / x.max(dim)</code> &nbsp; x.max() is the largest element (a 0-dimensional tensor); x.max(dim) works along one dimension and returns two tensors, the maximum values and their positions (indices). NumPy arrays have the same method: arr.max() and arr.max(axis=...), which returns only the values.</li></ul></details>

In [ ]:
def threshold_sweep(prob, target, thresholds):
    '''prob: foreground probabilities, target: 0 / 1, same shape (any); thresholds: a list of numbers ->
    (best_threshold, ious): the threshold with the highest IoU (a float), and the IoU of every threshold (array).'''
    prob = np.asarray(prob)
    t = np.asarray(target).astype(bool)                 # the true foreground
    ious = []
    for th in thresholds:
        # 1. the predicted foreground at this threshold
        p = prob >= th
        # 2. union and intersection, in pixels
        union = np.logical_or(p, t).sum()
        inter = np.logical_and(p, t).sum()
        # 3. the IoU (both empty: perfect agreement)
        if union > 0:
            ious.append(inter / union)
        else:
            ious.append(1.0)
    ious = np.array(ious)                               # (number of thresholds,)
    # 4. the index of the largest IoU
    best = ious.argmax()
    # 5. that threshold, and all the IoUs
    return float(thresholds[best]), ious

tests.check_threshold_sweep(threshold_sweep)

In [ ]:
ths = np.linspace(0.05, 0.95, 37)
best_t, ious_t = threshold_sweep(probs_fg, gts_fg, ths)
fig, ax = plt.subplots(figsize=(7, 3.8))
ax.plot(ths, ious_t, marker="o", ms=3)
ax.axvline(0.5, ls=":", color="grey", label=f"0.5: IoU {ious_t[np.argmin(np.abs(ths - 0.5))]:.3f}")
ax.axvline(best_t, color="#E45756", label=f"best {best_t:.2f}: IoU {ious_t.max():.3f}")
ax.set_xlabel("threshold on P(foreground)"); ax.set_ylabel("foreground IoU (validation)"); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 7 · Summary

* U-Net = encoder (double conv + pool) -> bottleneck -> decoder (x2 up + **concat skip** + double conv) -> 1x1 head.
* Channels double while the resolution halves; the input side must be divisible by $2^{\text{depth}}$.
* Binary: 1 channel + sigmoid + BCE + Dice; multi-class: C channels + softmax + CE (+ Dice, + class weights).
* Removing the skips (same budget) destroys the boundaries: the bottleneck knows *what*, the skips know *where*.
* Pretrained encoders + a light decoder are the default on small real datasets; read error maps, not just mIoU.
* Inference: sliding windows with averaged overlaps, flip TTA, thresholds tuned on validation.

### Self-check (answers in the Day 3 quiz)
1. A 4-level U-Net receives a 100x100 image. What goes wrong, and how do you fix it?
2. Why is the decoder's `ConvTranspose2d(k=2, s=2)` free of checkerboard artefacts?
3. Which loss and output activation for a mask where several classes can overlap at the same pixel?
4. Your U-Net predicts blobby, slightly shifted masks and misses small objects. Which component would you check first?
5. What does the diffusion U-Net add to the segmentation U-Net?

### Go deeper
* Ronneberger, Fischer & Brox, *U-Net: Convolutional Networks for Biomedical Image Segmentation* (MICCAI 2015).
* Zhou et al., *UNet++: A Nested U-Net Architecture* (2018); Oktay et al., *Attention U-Net* (2018).
* Chen et al., *Encoder-Decoder with Atrous Separable Convolution* (DeepLabv3+, ECCV 2018).
* Xie et al., *SegFormer* (NeurIPS 2021); Cheng et al., *Masked-attention Mask Transformer* (Mask2Former, CVPR 2022).
* Ho, Jain & Abbeel, *Denoising Diffusion Probabilistic Models* (2020): the U-Net as a denoiser.
* `segmentation_models_pytorch` documentation (encoders, losses, metrics).